In [ ]:
# Necessary Imports
import os
import gc
import time
import sys

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import numpy as np
import h5py # For load_mat_data
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, cohen_kappa_score

import matplotlib.pyplot as plt
# %matplotlib inline # Uncomment if running in a Jupyter environment

# --- Configuration ---
MAT_FILE_PATH = "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38.mat" #  MODIFIED: Update with your MAT file path
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RANDOM_STATE = 42
NUM_EPOCHS_DEMO = 5 # Keep epochs low for quick comparison (was 5, can be increased)
BATCH_SIZE = 256    # Adjust based on your GPU memory

# Assuming 102 raw dimensions in one-hot labels, where 0 is BG
NUM_ACTUAL_CLASSES = 101        # When BG is processed (ignored), we have 101 actual classes (0-100)
NUM_TOTAL_CLASSES_WITH_BG = 102 # When BG is NOT processed, we have 102 classes (0-101, where 0 is BG)

print(f"Using device: {DEVICE}")
print(f"MAT File Path: {MAT_FILE_PATH}")
if not os.path.exists(MAT_FILE_PATH):
    print(f"WARNING: MAT file not found at {MAT_FILE_PATH}. Please update the path.")

# --- 1. MAT Data Loading Utilities (adapted from brain_voxel_comparison.py) ---
def load_mat_data_from_h5(mat_file_path):
    """Simplified MAT loader from h5py."""
    print(f"Loading MAT data from: {mat_file_path}")
    if not os.path.exists(mat_file_path):
        raise FileNotFoundError(f"MAT file not found at {mat_file_path}")
    arrays = {}
    with h5py.File(mat_file_path, 'r') as f:
        for k, v in f.items():
            arrays[k] = np.array(v)
    print("MAT data loaded.")
    return arrays

def check_and_transpose_data_fn(arrays):
    """Checks and transposes MAT data."""
    print("Checking and transposing data...")
    data_key, region_key, prob_idx_key = 'data', 'region', 'prob_idx'

    # Basic check for key existence
    required_keys = [data_key, region_key, prob_idx_key]
    for key in required_keys:
        if key not in arrays:
            print(f"Error: Standard key '{key}' not found. Available keys: {list(arrays.keys())}")
            # Attempt to find alternative keys if common ones are missing
            # Example: some .mat files might use 'TD' for 'data', 'TR' for 'region', etc.
            # This part would need more sophisticated logic or user input if keys vary widely.
            # For now, we'll assume specific alternative or raise error.
            if key == 'data' and 'TRAIN_DATA' in arrays: data_key = 'TRAIN_DATA'
            elif key == 'region' and 'TRAIN_REGION' in arrays: region_key = 'TRAIN_REGION'
            elif key == 'prob_idx' and 'PROB_IDX' in arrays: prob_idx_key = 'PROB_IDX'
            else:
                raise KeyError(f"Could not find essential data key similar to '{key}' in .mat file. Available keys: {list(arrays.keys())}")
            print(f"Using alternative key: '{key}' -> '{eval(key + '_key')}'")


    # Data: (Features, Samples) -> (Samples, Features)
    # Assuming 341 features for this dataset
    expected_feature_dim = 341
    if arrays[data_key].shape[0] == expected_feature_dim:
        data_transposed = arrays[data_key].T
        print(f"Transposed '{data_key}' from {arrays[data_key].shape} to {data_transposed.shape}")
    elif arrays[data_key].shape[1] == expected_feature_dim:
        data_transposed = arrays[data_key]
        print(f"'{data_key}' already in correct dimension: {data_transposed.shape}")
    else:
        raise ValueError(f"Unexpected shape for '{data_key}': {arrays[data_key].shape}. Expected one dimension to be {expected_feature_dim}.")

    # Region: (Classes, Samples) -> (Samples, Classes)
    if arrays[region_key].shape[0] == NUM_TOTAL_CLASSES_WITH_BG: # 102 classes including BG
        region_transposed = arrays[region_key].T
        print(f"Transposed '{region_key}' from {arrays[region_key].shape} to {region_transposed.shape}")
    elif arrays[region_key].shape[1] == NUM_TOTAL_CLASSES_WITH_BG:
        region_transposed = arrays[region_key]
        print(f"'{region_key}' already in correct dimension: {region_transposed.shape}")
    else:
        raise ValueError(f"Unexpected shape for '{region_key}': {arrays[region_key].shape}. Expected one dimension to be {NUM_TOTAL_CLASSES_WITH_BG}.")

    prob_idx_transposed = arrays[prob_idx_key].flatten()
    print(f"Flattened '{prob_idx_key}' to shape: {prob_idx_transposed.shape}")


    assert data_transposed.shape[0] == region_transposed.shape[0] == prob_idx_transposed.shape[0], \
        f"Sample count mismatch after transpose/flatten: data({data_transposed.shape[0]}), region({region_transposed.shape[0]}), prob_idx({prob_idx_transposed.shape[0]})"
    assert data_transposed.shape[1] == expected_feature_dim, f"Feature count mismatch: expected {expected_feature_dim}, got {data_transposed.shape[1]}"
    assert region_transposed.shape[1] == NUM_TOTAL_CLASSES_WITH_BG, f"Class count mismatch: expected {NUM_TOTAL_CLASSES_WITH_BG}, got {region_transposed.shape[1]}"
    print("Data transposed and validated.")
    return data_transposed, region_transposed, prob_idx_transposed


def split_data_fn(data_len, prob_idx, random_state=RANDOM_STATE): # Pass data_len instead of full data
    """Splits data based on patient IDs using indices."""
    print("Splitting data by patient ID...")
    all_indices = np.arange(data_len)

    # Patient 38 for validation
    val_mask = (prob_idx == 38)
    val_indices = all_indices[val_mask]

    # Others for train/test pool
    train_test_pool_mask = ~val_mask
    train_test_pool_indices = all_indices[train_test_pool_mask]

    if len(train_test_pool_indices) == 0 and len(val_indices) > 0:
        print("Warning: No samples available for training/testing after selecting patient 38 for validation.")
        # Fallback: use some validation samples for training/testing if train_test_pool is empty
        # This is an edge case and indicates issues with the dataset or splitting logic for this specific dataset.
        # For now, we'll proceed, but this would likely lead to poor training.
        train_indices = np.array([], dtype=int)
        test_indices = np.array([], dtype=int)
    elif len(train_test_pool_indices) > 0:
        train_indices, test_indices = train_test_split(
            train_test_pool_indices, test_size=0.01, random_state=random_state, shuffle=True
        )
    else: # No data at all or only validation patient
        train_indices = np.array([], dtype=int)
        test_indices = np.array([], dtype=int)


    print(f"Total samples: {data_len}")
    print(f"Train indices: {len(train_indices)}, Val indices: {len(val_indices)}, Test indices: {len(test_indices)}")
    if data_len > 0 :
        print(f"  Train: {len(train_indices)/data_len*100:.2f}%, Val: {len(val_indices)/data_len*100:.2f}%, Test: {len(test_indices)/data_len*100:.2f}%")

    # Sanity check
    if len(train_indices) == 0 and NUM_EPOCHS_DEMO > 0:
        print("CRITICAL WARNING: Training set is empty. Training will likely fail or produce meaningless results.")
        # Consider raising an error if training is expected:
        # raise ValueError("Training set is empty after splitting. Check patient IDs and data.")

    return train_indices, val_indices, test_indices

# --- 2. Scaler Creation ---
def create_scaler_fn(all_data, train_indices):
    """Creates and fits StandardScaler on training data."""
    print("Creating and fitting StandardScaler on training data...")
    scaler = StandardScaler()
    if len(train_indices) == 0:
        print("Warning: Training set is empty, scaler will not be fitted. Using default (identity) scaler.")
        # scaler.mean_ = np.zeros(all_data.shape[1]) # Assuming all_data has correct feature dim
        # scaler.scale_ = np.ones(all_data.shape[1])
        return scaler # Return unfitted scaler
    
    print(f"Fitting scaler on {len(train_indices)} training samples.")
    scaler.fit(all_data[train_indices])
    print("StandardScaler fitted.")
    return scaler

# --- 3. Custom PyTorch Dataset ---
class BrainVoxelPyTorchDataset(Dataset):
    def __init__(self, all_features_np, all_one_hot_labels_np, indices_np, scaler_obj,
                 process_background=True):
        
        if len(indices_np) == 0:
            print("Dataset Warning: Initializing with zero samples.")
            self.features_scaled_np = np.array([]).reshape(0, all_features_np.shape[1] if all_features_np.ndim > 1 else 0)
            self.processed_labels_np = np.array([], dtype=np.int64)
        else:
            self.features_np = all_features_np[indices_np]
            self.one_hot_labels_np = all_one_hot_labels_np[indices_np]
            self.scaler = scaler_obj

            # Scale features
            if hasattr(self.scaler, 'mean_') and self.scaler.mean_ is not None: # Check if scaler is fitted
                self.features_scaled_np = self.scaler.transform(self.features_np)
            else:
                print("Dataset Warning: Scaler is not fitted. Using raw features.")
                self.features_scaled_np = self.features_np


            # Process labels
            # Convert one-hot to indexed labels (0 to 101, where 0 is original BG)
            indexed_raw_labels_np = np.argmax(self.one_hot_labels_np, axis=1)

            if process_background:
                # BG (original index 0) -> -1 (for ignore_index)
                # Actual classes (original indices 1-101) -> mapped to 0-100
                self.processed_labels_np = indexed_raw_labels_np - 1 # Map 1-101 to 0-100; 0 to -1
                # Explicitly ensure original index 0 (BG) becomes -1
                self.processed_labels_np[indexed_raw_labels_np == 0] = -1
            else:
                # BG is treated as class 0, actual classes are 1-101
                self.processed_labels_np = indexed_raw_labels_np
        
        print(f"Dataset created. Samples: {len(self.processed_labels_np)}. BG processed: {process_background}.")
        if len(self.processed_labels_np) > 0:
             print(f"  Label range: {np.min(self.processed_labels_np) if len(self.processed_labels_np)>0 else 'N/A'} to {np.max(self.processed_labels_np) if len(self.processed_labels_np)>0 else 'N/A'}")


    def __len__(self):
        return len(self.processed_labels_np)

    def __getitem__(self, idx):
        features = torch.FloatTensor(self.features_scaled_np[idx])
        # Squeeze to make it a scalar tensor if it's a single value array
        label = torch.LongTensor([self.processed_labels_np[idx]]).squeeze()
        return features, label

# --- 4. Model Definition (FixedMLP) ---
class FixedMLP(nn.Module):
    def __init__(self, input_dim=341, num_output_classes=101, dropout_rate=0.5, l2_reg=1e-5):
        super(FixedMLP, self).__init__()
        self.l2_reg = l2_reg
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 4096), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(4096, 4096), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(4096, 4096), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(4096, 4096), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(4096, num_output_classes)
        )
        for layer in self.layers: # Weight initialization
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)

    def forward(self, x):
        return self.layers(x)

    def get_l2_loss(self):
        l2_loss = torch.tensor(0., device=DEVICE) # Ensure tensor is on the correct device
        for param in self.parameters():
            if param.requires_grad:
                 l2_loss = l2_loss + torch.norm(param, 2) ** 2
        return self.l2_reg * l2_loss

# --- 5. Robust Class Weight Calculation ---
def calculate_weights_fn_robust(labels_for_weights_np, num_effective_classes, process_background_flag, smoothing_epsilon=1e-9):
    """
    Calculates class weights robustly.
    labels_for_weights_np: 1D numpy array of processed labels (-1 for BG if process_background_flag, 0 to N-1 otherwise)
    num_effective_classes: Number of classes the model predicts (e.g., 101 or 102)
    process_background_flag: Boolean indicating if background was processed to -1
    """
    print(f"Calculating class weights for {num_effective_classes} effective classes. BG processed: {process_background_flag}")
    
    if len(labels_for_weights_np) == 0:
        print("Warning: No labels provided for weight calculation. Returning equal weights.")
        return torch.ones(num_effective_classes, dtype=torch.float32).to(DEVICE)

    if process_background_flag:
        # Labels are -1 (BG), 0 to (num_effective_classes-1) for actual classes
        actual_class_labels_np = labels_for_weights_np[labels_for_weights_np >= 0]
        if len(actual_class_labels_np) == 0:
            print("Warning: No valid (non-background) class samples found for weight calculation. Returning equal weights for effective classes.")
            return torch.ones(num_effective_classes, dtype=torch.float32).to(DEVICE)
        counts_np = np.bincount(actual_class_labels_np, minlength=num_effective_classes)
    else:
        # Labels are 0 (BG as a class) to (num_effective_classes-1)
        counts_np = np.bincount(labels_for_weights_np, minlength=num_effective_classes)

    # Inverse frequency weighting
    # Add epsilon to counts to prevent division by zero for classes not in labels_for_weights_np
    # (e.g., a class exists in val but not in train subset used for weight calc)
    weights_np = 1.0 / (counts_np + smoothing_epsilon)
    
    # For classes truly absent in the training labels (counts_np[i] == 0),
    # their weight will be 1.0/smoothing_epsilon, which is very high.
    # It might be better to give them a moderate weight, e.g., 1.0 or max of other weights.
    # Let's assign a weight of 1.0 for classes with zero count in the provided labels.
    weights_np[counts_np == 0] = 1.0 
    
    # Optional: Normalize weights so that they sum up to num_effective_classes or similar
    # weights_np = weights_np / np.sum(weights_np) * num_effective_classes # Normalization example

    print(f"  Class counts (first few/total {len(counts_np)}): {counts_np[:min(10, len(counts_np))]}")
    print(f"  Calculated weights (first few/total {len(weights_np)}): {weights_np[:min(10, len(weights_np))]}")
    if np.any(counts_np > 0):
         min_w_present = np.min(weights_np[counts_np > 0])
         max_w_present = np.max(weights_np[counts_np > 0])
         print(f"  Weight range (for classes present in labels): [{min_w_present:.4f} - {max_w_present:.4f}]")
    else:
        print("  No classes were present in the labels provided for weight calculation.")
    print(f"  Weights for classes with 0 count set to: {weights_np[counts_np == 0][0] if np.any(counts_np==0) else 'N/A (all classes present)'}")

    return torch.FloatTensor(weights_np).to(DEVICE)


# --- 6. Training and Evaluation Loops ---
def train_epoch_fn(model, dataloader, criterion, optimizer, device, add_l2_loss_flag=True):
    model.train()
    running_loss = 0.0
    total_samples = 0
    all_preds_list = []
    all_true_list = []

    for features, labels in dataloader:
        if features.nelement() == 0 : continue # Skip empty batches
        features, labels = features.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(features)
        
        classification_loss = criterion(outputs, labels)
        
        final_loss = classification_loss
        if add_l2_loss_flag and hasattr(model, 'get_l2_loss'):
            l2_loss = model.get_l2_loss()
            final_loss = final_loss + l2_loss

        if torch.isnan(final_loss):
            print("NaN loss encountered during training!")
            print(f"Features (sample sum): {features[0].sum() if features.nelement() > 0 else 'N/A'}")
            print(f"Labels (sample): {labels[0] if labels.nelement() > 0 else 'N/A'}")
            print(f"Outputs (sample): {outputs[0] if outputs.nelement() > 0 else 'N/A'}")
            # Optionally raise error or skip batch
            continue 


        final_loss.backward()
        optimizer.step()
        
        running_loss += final_loss.item() * features.size(0)
        total_samples += features.size(0)
        
        # Store predictions and true labels for metrics
        # Only consider non-ignored labels for metrics if ignore_index is used
        active_labels_mask = (labels != criterion.ignore_index) if criterion.ignore_index is not None else torch.ones_like(labels, dtype=torch.bool)
        
        if active_labels_mask.sum() > 0:
            preds = torch.argmax(outputs[active_labels_mask], dim=1)
            all_preds_list.extend(preds.cpu().numpy())
            all_true_list.extend(labels[active_labels_mask].cpu().numpy())
            
    if total_samples == 0: return 0.0, 0.0, 0.0 # Avoid division by zero
    
    epoch_loss = running_loss / total_samples
    epoch_acc = accuracy_score(all_true_list, all_preds_list) if len(all_true_list) > 0 else 0.0
    # Use labels parameter for f1_score to handle cases where not all classes are present.
    # Determine the set of unique labels present in true or preds for 'labels' param of f1_score
    unique_lbls = np.unique(np.concatenate((all_true_list, all_preds_list))) if len(all_true_list)>0 else np.array([])
    if criterion.ignore_index is not None:
        unique_lbls = unique_lbls[unique_lbls != criterion.ignore_index]

    epoch_f1 = f1_score(all_true_list, all_preds_list, labels=unique_lbls if len(unique_lbls)>0 else None, average='macro', zero_division=0) if len(all_true_list) > 0 else 0.0
    
    return epoch_loss, epoch_acc, epoch_f1


def evaluate_model_fn(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    total_samples = 0
    all_preds_list = []
    all_true_list = []

    with torch.no_grad():
        for batch_idx, (features, labels) in enumerate(dataloader):
            if features.nelement() == 0 : continue # Skip empty batches
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            
            # For validation, usually just classification loss, not L2 part from model
            loss = criterion(outputs, labels) 

            if torch.isnan(loss):
                print(f"ERROR: NaN VAL loss detected in Val Batch {batch_idx}!")
                # --- DETAILED DEBUG PRINT (Uncomment to use) ---
                # print(f"  Problematic Features (norm): {torch.norm(features).item()}")
                # print(f"  Problematic Features (sum): {features.sum().item()}")
                # print(f"  Problematic Labels: {labels.cpu().numpy()}")
                # print(f"  Unique Labels in problematic batch: {torch.unique(labels).cpu().numpy()}")
                # print(f"  Model Outputs (sample of first 5, first 10 logits): \n{outputs[:5, :10].cpu().numpy()}")
                # print(f"  Criterion ignore_index: {criterion.ignore_index}")
                # print(f"  Criterion weights (sum, min, max if exists): { (criterion.weight.sum().item(), criterion.weight.min().item(), criterion.weight.max().item()) if criterion.weight is not None else 'No weights'}")
                # Check if all labels are ignored
                # if criterion.ignore_index is not None and torch.all(labels == criterion.ignore_index):
                #     print("  VAL BATCH WARNING: All labels in this batch are ignore_index.")
                # torch.save({
                #     'features': features.cpu(), 'labels': labels.cpu(), 
                #     'outputs': outputs.cpu(), 'loss_val': loss.item()
                #     }, f'nan_val_batch_{batch_idx}.pt')
                # print(f"  Saved problematic batch to nan_val_batch_{batch_idx}.pt")
                # --- END DETAILED DEBUG ---
                # To stop on NaN: raise RuntimeError("NaN loss in validation")
                # To continue but flag: return float('nan'), float('nan'), float('nan') # Propagate NaN
                # For now, we will let it contribute to running_loss if it's a number, 
                # but if loss itself is NaN, running_loss becomes NaN.
            
            if not torch.isnan(loss): # Only add if not NaN
                running_loss += loss.item() * features.size(0)
            else: # if loss is NaN, the whole epoch loss will be NaN
                running_loss = float('nan') # Propagate NaN
                
            total_samples += features.size(0)

            active_labels_mask = (labels != criterion.ignore_index) if criterion.ignore_index is not None else torch.ones_like(labels, dtype=torch.bool)
            if active_labels_mask.sum() > 0:
                preds = torch.argmax(outputs[active_labels_mask], dim=1)
                all_preds_list.extend(preds.cpu().numpy())
                all_true_list.extend(labels[active_labels_mask].cpu().numpy())
    
    if total_samples == 0 or np.isnan(running_loss): # check for NaN in running_loss
        return float('nan') if np.isnan(running_loss) else 0.0, 0.0, 0.0

    epoch_loss = running_loss / total_samples
    epoch_acc = accuracy_score(all_true_list, all_preds_list) if len(all_true_list) > 0 else 0.0
    unique_lbls = np.unique(np.concatenate((all_true_list, all_preds_list))) if len(all_true_list)>0 else np.array([])
    if criterion.ignore_index is not None:
        unique_lbls = unique_lbls[unique_lbls != criterion.ignore_index]
        
    epoch_f1 = f1_score(all_true_list, all_preds_list, labels=unique_lbls if len(unique_lbls)>0 else None, average='macro', zero_division=0) if len(all_true_list) > 0 else 0.0
    
    return epoch_loss, epoch_acc, epoch_f1

# --- Load and Prepare Full Dataset (once) ---
print("--- Initial Data Loading and Preparation ---")
try:
    mat_arrays = load_mat_data_from_h5(MAT_FILE_PATH)
    all_features_raw_np, all_labels_one_hot_raw_np, prob_idx_raw_np = check_and_transpose_data_fn(mat_arrays)
    train_idx_np, val_idx_np, test_idx_np = split_data_fn(len(all_features_raw_np), prob_idx_raw_np)

    # Create scaler based on training data
    scaler_obj = create_scaler_fn(all_features_raw_np, train_idx_np)
    print(f"Scaler mean (first 5 features): {scaler_obj.mean_[:5] if hasattr(scaler_obj, 'mean_') and scaler_obj.mean_ is not None else 'Scaler not fitted'}")

except FileNotFoundError as e:
    print(f"ERROR during initial data load: {e}")
    print("Please ensure MAT_FILE_PATH is correct and the file exists.")
    # Exit or skip scenarios if data can't be loaded
    sys.exit("Data loading failed.")
except KeyError as e:
    print(f"ERROR during initial data load (KeyError): {e}")
    print("Please check the keys in your .mat file and adjust 'check_and_transpose_data_fn' if necessary.")
    sys.exit("Data loading failed due to missing keys.")
except ValueError as e:
    print(f"ERROR during initial data load (ValueError): {e}")
    sys.exit("Data loading failed due to value error (e.g. unexpected shapes, empty training set).")


# Store results
results_summary = {}
history_plots = {} # To store history for plotting

# --- Helper function to run a scenario ---
def run_scenario(scenario_name, process_bg, use_weights, 
                 all_features_np, all_labels_one_hot_np, 
                 train_indices_np, val_indices_np, scaler,
                 num_epochs, batch_size, device):
    print(f"\n\n--- Running {scenario_name} ---")
    print(f"Process Background: {process_bg}, Use Class Weights: {use_weights}")

    # Determine number of output classes for the model
    num_model_output_classes = NUM_ACTUAL_CLASSES if process_bg else NUM_TOTAL_CLASSES_WITH_BG

    # DataLoaders
    train_dataset = BrainVoxelPyTorchDataset(all_features_np, all_labels_one_hot_np, train_indices_np, scaler, process_background=process_bg)
    val_dataset = BrainVoxelPyTorchDataset(all_features_np, all_labels_one_hot_np, val_indices_np, scaler, process_background=process_bg)
    
    if len(train_dataset) == 0:
        print(f"WARNING for {scenario_name}: Training dataset is empty. Skipping training.")
        results_summary[scenario_name] = {'val_acc': 0.0, 'val_f1': 0.0, 'history': {k:[] for k in ['train_loss', 'train_acc', 'train_f1', 'val_loss', 'val_acc', 'val_f1']}}
        history_plots[scenario_name] = results_summary[scenario_name]['history']
        return

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0) # num_workers=0 for main thread
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    # Model
    model = FixedMLP(num_output_classes=num_model_output_classes).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-5) # L2 is manually added in train_epoch_fn

    # Criterion
    class_weights_tensor = None
    if use_weights:
        # Pass processed labels from the dataset object for weight calculation
        class_weights_tensor = calculate_weights_fn_robust(
            train_dataset.processed_labels_np, # Use the already processed labels from dataset
            num_model_output_classes, # Num classes model is trying to predict
            process_background_flag=process_bg
        )
    
    ignore_idx = -1 if process_bg else -999 # Use an unused value if not ignoring, effectively no ignore
    if process_bg:
        criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, ignore_index=ignore_idx)
    else: # Not processing background (BG is class 0)
        criterion = nn.CrossEntropyLoss(weight=class_weights_tensor) # No ignore_index, or set to a value not in labels

    print(f"  Model output classes: {num_model_output_classes}")
    print(f"  Criterion ignore_index: {criterion.ignore_index if process_bg else 'None (BG is class 0)'}")
    print(f"  Using class weights: {use_weights} " + (f"(sum: {class_weights_tensor.sum().item():.2f})" if class_weights_tensor is not None else ""))


    history = {'train_loss':[], 'train_acc':[], 'train_f1':[], 'val_loss':[], 'val_acc':[], 'val_f1':[]}
    best_val_f1 = -1.0

    for epoch in range(num_epochs):
        start_e_time = time.time()
        train_loss, train_acc, train_f1 = train_epoch_fn(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, val_f1 = evaluate_model_fn(model, val_loader, criterion, device)
        
        history['train_loss'].append(train_loss); history['train_acc'].append(train_acc); history['train_f1'].append(train_f1)
        history['val_loss'].append(val_loss); history['val_acc'].append(val_acc); history['val_f1'].append(val_f1)
        
        if np.isnan(val_f1) or np.isnan(val_acc):
            print(f"{scenario_name} - Epoch {epoch+1}/{num_epochs} -> Val F1/Acc is NaN. Stopping scenario.")
            break # Stop this scenario if validation results in NaN
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            # Could save best model state here if needed

        print(f"{scenario_name} - Epoch {epoch+1}/{num_epochs} ({(time.time()-start_e_time):.2f}s) -> "
              f"Train L: {train_loss:.4f}, A: {train_acc:.4f}, F1: {train_f1:.4f} | "
              f"Val L: {val_loss:.4f}, A: {val_acc:.4f}, F1: {val_f1:.4f}")
              
    results_summary[scenario_name] = {'val_acc': history['val_acc'][-1] if history['val_acc'] else 0.0,
                                      'val_f1': history['val_f1'][-1] if history['val_f1'] else 0.0,
                                      'history': history}
    history_plots[scenario_name] = history # Store for plotting
    
    del model, train_dataset, val_dataset, train_loader, val_loader, criterion, optimizer
    if class_weights_tensor is not None: del class_weights_tensor
    gc.collect()
    if device.type == 'cuda': torch.cuda.empty_cache()



In [ ]:

# --- Run the Four Scenarios ---
# Scenario 1: WITH Background Processing, WITH Class Weights
run_scenario("S1 (WithBG, WithW)", process_bg=True, use_weights=True,
             all_features_np=all_features_raw_np, all_labels_one_hot_np=all_labels_one_hot_raw_np,
             train_indices_np=train_idx_np, val_indices_np=val_idx_np, scaler=scaler_obj,
             num_epochs=NUM_EPOCHS_DEMO, batch_size=BATCH_SIZE, device=DEVICE)


In [ ]:
# Scenario 2: WITHOUT Background Processing, WITH Class Weights
run_scenario("S2 (NoBG, WithW)", process_bg=False, use_weights=True,
             all_features_np=all_features_raw_np, all_labels_one_hot_np=all_labels_one_hot_raw_np,
             train_indices_np=train_idx_np, val_indices_np=val_idx_np, scaler=scaler_obj,
             num_epochs=NUM_EPOCHS_DEMO, batch_size=BATCH_SIZE, device=DEVICE)

In [ ]:

# Scenario 3: WITH Background Processing, WITHOUT Class Weights
run_scenario("S3 (WithBG, NoW)", process_bg=True, use_weights=False,
             all_features_np=all_features_raw_np, all_labels_one_hot_np=all_labels_one_hot_raw_np,
             train_indices_np=train_idx_np, val_indices_np=val_idx_np, scaler=scaler_obj,
             num_epochs=NUM_EPOCHS_DEMO, batch_size=BATCH_SIZE, device=DEVICE)


In [ ]:

# Scenario 4: WITHOUT Background Processing, WITHOUT Class Weights
run_scenario("S4 (NoBG, NoW)", process_bg=False, use_weights=False,
             all_features_np=all_features_raw_np, all_labels_one_hot_np=all_labels_one_hot_raw_np,
             train_indices_np=train_idx_np, val_indices_np=val_idx_np, scaler=scaler_obj,
             num_epochs=NUM_EPOCHS_DEMO, batch_size=BATCH_SIZE, device=DEVICE)

In [ ]:

# --- 7. Results Comparison ---
print("\n\n--- Final Results Summary ---")
print(f"{'Scenario':<40} | {'Final Val Accuracy':<20} | {'Final Val Macro F1':<20}")
print("-" * 85)
for scenario, metrics in results_summary.items():
    # Check if history exists and is not empty
    val_acc_display = metrics['history']['val_acc'][-1] if metrics.get('history') and metrics['history']['val_acc'] else 'N/A'
    val_f1_display = metrics['history']['val_f1'][-1] if metrics.get('history') and metrics['history']['val_f1'] else 'N/A'

    if isinstance(val_acc_display, float): val_acc_display = f"{val_acc_display:.4f}"
    if isinstance(val_f1_display, float): val_f1_display = f"{val_f1_display:.4f}"

    print(f"{scenario:<40} | {val_acc_display:<20} | {val_f1_display:<20}")


# --- Plotting training curves ---
num_scenarios = len(history_plots)
if num_scenarios > 0:
    fig, axes = plt.subplots(2, 1, figsize=(14, 12)) # Two plots: F1 and Accuracy

    # Plot Validation F1-Score
    for scenario_name, hist_data in history_plots.items():
        if hist_data and hist_data.get('val_f1'): # Check if history and val_f1 data exist
             # Filter out NaNs for plotting if any scenario was stopped early due to NaN
            epochs_run = np.arange(1, len(hist_data['val_f1']) + 1)
            valid_f1_scores = np.array(hist_data['val_f1'])
            
            axes[0].plot(epochs_run, valid_f1_scores, marker='o', linestyle='-', label=f"{scenario_name}")

    axes[0].set_title(f'Validation Macro F1-Score Comparison (Epochs: {NUM_EPOCHS_DEMO})')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Macro F1-Score')
    axes[0].legend(loc='center left', bbox_to_anchor=(1, 0.5))
    axes[0].grid(True)
    axes[0].set_xticks(np.arange(1, NUM_EPOCHS_DEMO + 1))


    # Plot Validation Accuracy
    for scenario_name, hist_data in history_plots.items():
        if hist_data and hist_data.get('val_acc'): # Check if history and val_acc data exist
            epochs_run = np.arange(1, len(hist_data['val_acc']) + 1)
            valid_acc_scores = np.array(hist_data['val_acc'])

            axes[1].plot(epochs_run, valid_acc_scores, marker='x', linestyle='--', label=f"{scenario_name}")

    axes[1].set_title(f'Validation Accuracy Comparison (Epochs: {NUM_EPOCHS_DEMO})')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend(loc='center left', bbox_to_anchor=(1, 0.5))
    axes[1].grid(True)
    axes[1].set_xticks(np.arange(1, NUM_EPOCHS_DEMO + 1))


    plt.tight_layout(rect=[0, 0, 0.85, 1]) # Adjust layout to make space for legend
    plt.show()
else:
    print("No history data to plot. Did the scenarios run correctly?")